# 🌸 Flower Image Colorization with a Conditional GAN (pix2pix)

This notebook trains a **conditional GAN** that turns a grayscale flower drawing into a colour one.

* **Generator:** a U-Net (encoder–decoder with skip connections). It maps a 1-channel gray image to a 3-channel RGB image.
* **Discriminator:** a PatchGAN. It sees the *(gray, colour)* pair and scores each image patch as real or fake.
* **Objective:** adversarial loss + λ·L1 reconstruction loss, as in [pix2pix (Isola et al., 2017)](https://arxiv.org/abs/1611.07004).

**Run it on Google Colab with a GPU runtime** (*Runtime → Change runtime type → T4 GPU*), then *Run all*.
Project terms (species, stem, angle block, …) are defined in `CONTEXT.md`.

## 1. Setup

In [ ]:
import os
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

%matplotlib inline

In [ ]:
# ---- Configuration -------------------------------------------------------
DATA_DIR = "flower_dataset"
FLOWER_SPECIES = ("azalea", "daisy", "marigold", "orchid", "rose")

SEED = 42
IMAGE_SIZE = 128          # native resolution of the dataset
SUBSET_FRACTION = 0.2     # fraction of the 21,600 variants used (keeps a Colab run short)
ANGLE_BLOCK = 15          # degrees of folded angle per block used for the train/test split
SPLIT_GAP = 5             # training images within this many degrees of a test angle are dropped
TEST_FRACTION = 0.2

BATCH_SIZE = 16
EPOCHS = 50
CHECKPOINT_PATH = "checkpoint.pth"
LR = 2e-4
BETAS = (0.5, 0.999)
L1_LAMBDA = 100           # weight of the L1 term in the generator loss (pix2pix default)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Download the dataset

The archive holds 128×128 RGBA PNGs of hand-drawn flowers on a transparent background.

In [ ]:
if not os.path.isdir(DATA_DIR):
    !gdown 1RbxAgZOqQ_NUPHCeo8ryP_ibRQpnm6Li
    !unzip -qo flower_dataset.zip
print(len(os.listdir(DATA_DIR)), "files in", DATA_DIR)

## 3. Index the images

Filenames follow `<species>_<angle>_core<stem>[_flip].png`:

| Part | Meaning |
| --- | --- |
| `species` | azalea, daisy, marigold, orchid, rose |
| `angle` | rotation of the flower head, 0–359° |
| `core1`–`core6` | one of six stem shapes |
| `_flip` | horizontal mirror |

That gives 5 × 360 × 6 × 2 = **21,600** variants.
The archive also contains `sunflower_*` and `.DS_S_*` files. These are **byte-identical copies of the `rose` images** (an export glitch), so we drop them along with `.DS_Store`.

In [ ]:
FILENAME_RE = re.compile(r"^(?P<species>[a-z]+)_(?P<angle>\d+)_core(?P<stem>\d)(?P<flip>_flip)?\.png$")

records = []
for name in sorted(os.listdir(DATA_DIR)):
    m = FILENAME_RE.match(name)
    if m and m["species"] in FLOWER_SPECIES:
        records.append({
            "path": os.path.join(DATA_DIR, name),
            "species": m["species"],
            "angle": int(m["angle"]),
            "stem": int(m["stem"]),
            "flip": m["flip"] is not None,
        })

df = pd.DataFrame(records)
print(f"{len(df)} valid images (skipped {len(os.listdir(DATA_DIR)) - len(df)} duplicates/junk)")
df.groupby("species").size()

## 4. Subset and train/test split

* **Subset:** a random sample of `SUBSET_FRACTION`, stratified by species so every flower is represented equally.
* **Split by angle block:** images at neighbouring angles (and their stem/flip variants) are almost identical, so a random per-file split would leak near-copies of training images into the test set. Instead:
  * A flipped image at angle *a* looks like an unflipped one at *360 − a*. So we work with the **folded angle** `min(a, 360 − a)` (0–180°), which puts mirror twins in the same group.
  * Folded angles are grouped into `ANGLE_BLOCK`-degree blocks, and whole blocks are held out for testing.
  * Training images within `SPLIT_GAP` degrees of any test angle are dropped, so no training image is a near-rotation of a test image.

In [ ]:
subset, _ = train_test_split(df, train_size=SUBSET_FRACTION, stratify=df["species"], random_state=SEED)

folded = np.minimum(subset["angle"], 360 - subset["angle"])
groups = np.minimum(folded // ANGLE_BLOCK, 180 // ANGLE_BLOCK - 1)  # fold 180 into the last block
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED)
train_idx, test_idx = next(splitter.split(subset, groups=groups))

test_angles = np.unique(folded.iloc[test_idx])
gap = np.abs(folded.iloc[train_idx].to_numpy()[:, None] - test_angles[None, :]).min(axis=1)
train_idx = train_idx[gap > SPLIT_GAP]
train_df, test_df = subset.iloc[train_idx], subset.iloc[test_idx]

print(f"train: {len(train_df)}  test: {len(test_df)}")
print("test folded-angle blocks (degrees):",
      sorted(int(b) * ANGLE_BLOCK for b in np.unique(groups.iloc[test_idx])))

## 5. Dataset and DataLoaders

Each sample is a *(gray, colour)* pair, both scaled to [-1, 1] to match the generator's `tanh` output.
* The transparent background is composited onto **white**. A plain `convert("RGB")` would turn it black.
* Gray uses perceptual luminance (`0.299R + 0.587G + 0.114B`) rather than a plain channel mean.

In [ ]:
LUMA = torch.tensor([0.299, 0.587, 0.114]).view(3, 1, 1)


class FlowerColorizationDataset(Dataset):
    def __init__(self, paths, image_size=IMAGE_SIZE):
        self.paths = list(paths)
        self.image_size = image_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            rgba = im.convert("RGBA")
        background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
        rgb = Image.alpha_composite(background, rgba).convert("RGB")
        if rgb.size != (self.image_size, self.image_size):
            rgb = rgb.resize((self.image_size, self.image_size), Image.BICUBIC)

        color = torch.from_numpy(np.asarray(rgb, dtype=np.float32) / 255.0).permute(2, 0, 1)
        gray = (color * LUMA).sum(dim=0, keepdim=True)
        return gray * 2 - 1, color * 2 - 1


def to_image(t):
    # [-1, 1] tensor (C, H, W) -> [0, 1] numpy array (H, W, C) for plotting / metrics
    return (t.detach().cpu().clamp(-1, 1) * 0.5 + 0.5).permute(1, 2, 0).numpy()


train_loader = DataLoader(FlowerColorizationDataset(train_df["path"]), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, drop_last=True,
                          generator=torch.Generator().manual_seed(SEED))
test_loader = DataLoader(FlowerColorizationDataset(test_df["path"]), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)

# Preview straight from the dataset so the seeded training shuffle is untouched
preview_ds = train_loader.dataset
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i, idx in enumerate(np.random.default_rng(SEED).choice(len(preview_ds), 6, replace=False)):
    gray, color = preview_ds[idx]
    axes[0, i].imshow(to_image(gray).squeeze(-1), cmap="gray")
    axes[1, i].imshow(to_image(color))
for ax in axes.flat:
    ax.axis("off")
axes[0, 0].set_title("Gray input", loc="left")
axes[1, 0].set_title("Colour target", loc="left")
plt.tight_layout()
plt.show()

## 6. Model

### Generator: U-Net
Three down-sampling blocks, a bottleneck, and three up-sampling blocks. Each decoder level is concatenated with the matching encoder feature map (skip connection), which keeps the sharp line-art edges.

In [ ]:
class UNetBlock(nn.Module):
    # 4x4 conv with stride 2: halves (down) or doubles (up) the spatial size
    def __init__(self, in_channels, out_channels, down=True, use_dropout=False, norm=True):
        super().__init__()
        if down:
            self.block = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=not norm),
                nn.BatchNorm2d(out_channels) if norm else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True),
            )
        else:
            self.block = nn.Sequential(
                nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
        self.dropout = nn.Dropout(0.5) if use_dropout else nn.Identity()

    def forward(self, x):
        return self.dropout(self.block(x))


class UNetGenerator(nn.Module):
    def __init__(self, in_channels=1, out_channels=3):
        super().__init__()
        # Encoder (H = input size)
        self.down1 = UNetBlock(in_channels, 32, norm=False)  # H   -> H/2 (no norm on the first layer, as in pix2pix)
        self.down2 = UNetBlock(32, 64)                       # H/2 -> H/4
        self.down3 = UNetBlock(64, 128)                      # H/4 -> H/8

        self.bottleneck = nn.Sequential(          # H/8 -> H/16
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.ReLU(inplace=True),
        )

        # Decoder: input channels double because of the skip concatenation
        self.up1 = UNetBlock(256, 128, down=False, use_dropout=True)  # H/16 -> H/8
        self.up2 = UNetBlock(128 + 128, 64, down=False)               # H/8  -> H/4
        self.up3 = UNetBlock(64 + 64, 32, down=False)                 # H/4  -> H/2

        self.final = nn.Sequential(                                   # H/2  -> H
            nn.ConvTranspose2d(32 + 32, out_channels, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        b = self.bottleneck(d3)

        u1 = self.up1(b)
        u2 = self.up2(torch.cat([u1, d3], dim=1))
        u3 = self.up3(torch.cat([u2, d2], dim=1))
        return self.final(torch.cat([u3, d1], dim=1))

### Discriminator: PatchGAN
It takes the gray input and a colour image (real or generated), stacked into 4 channels. It outputs a grid of **logits**, one per overlapping patch. The sigmoid lives inside `BCEWithLogitsLoss`, which is numerically more stable than `Sigmoid` + `BCELoss`.

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=1 + 3):
        super().__init__()

        def block(c_in, c_out, stride, norm=True):
            layers = [nn.Conv2d(c_in, c_out, 4, stride, 1, bias=not norm)]
            if norm:
                layers.append(nn.BatchNorm2d(c_out))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_channels, 64, 2, norm=False),
            *block(64, 128, 2),
            *block(128, 256, 2),
            *block(256, 512, 1),
            nn.Conv2d(512, 1, 4, 1, 1),  # one logit per patch
        )

    def forward(self, gray_img, color_img):
        return self.model(torch.cat([gray_img, color_img], dim=1))


def init_weights(m):
    # pix2pix initialisation: N(0, 0.02) for conv weights, N(1, 0.02) for BatchNorm scale
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


gen = UNetGenerator().to(device)
disc = PatchDiscriminator().to(device)
gen.apply(init_weights)
disc.apply(init_weights)

# Shape sanity check
with torch.no_grad():
    x = torch.randn(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
    y = gen(x)
    print("generator out:    ", tuple(y.shape))
    print("discriminator out:", tuple(disc(x, y).shape), "(patch logits)")
print(f"G params: {sum(p.numel() for p in gen.parameters()):,}  |  D params: {sum(p.numel() for p in disc.parameters()):,}")

## 7. Training

Each step:
1. **Discriminator:** push real pairs toward 1 and generated pairs toward 0.
2. **Generator:** fool D (adversarial loss), plus stay close to the real colours (`L1_LAMBDA × L1`).

Without the L1 term, G only learns to fool D. Colours then drift and G's loss keeps climbing as D wins.
Losses are averaged over each epoch, not taken from the last batch.

In [ ]:
adversarial_loss = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()
opt_gen = torch.optim.Adam(gen.parameters(), lr=LR, betas=BETAS)
opt_disc = torch.optim.Adam(disc.parameters(), lr=LR, betas=BETAS)

history = {"d_loss": [], "g_adv": [], "g_l1": []}

for epoch in range(EPOCHS):
    gen.train()
    disc.train()
    totals = {k: 0.0 for k in history}

    for gray, real in train_loader:
        gray, real = gray.to(device), real.to(device)
        fake = gen(gray)

        # --- Discriminator ---
        d_real = disc(gray, real)
        d_fake = disc(gray, fake.detach())
        d_loss = (adversarial_loss(d_real, torch.ones_like(d_real)) +
                  adversarial_loss(d_fake, torch.zeros_like(d_fake))) / 2
        opt_disc.zero_grad()
        d_loss.backward()
        opt_disc.step()

        # --- Generator ---
        d_fake = disc(gray, fake)
        g_adv = adversarial_loss(d_fake, torch.ones_like(d_fake))
        g_l1 = l1_loss(fake, real)
        g_loss = g_adv + L1_LAMBDA * g_l1
        opt_gen.zero_grad()
        g_loss.backward()
        opt_gen.step()

        totals["d_loss"] += d_loss.item()
        totals["g_adv"] += g_adv.item()
        totals["g_l1"] += g_l1.item()

    for k in history:
        history[k].append(totals[k] / len(train_loader))
    print(f"Epoch [{epoch + 1:>3}/{EPOCHS}] | D loss: {history['d_loss'][-1]:.4f} | "
          f"G adv: {history['g_adv'][-1]:.4f} | G L1: {history['g_l1'][-1]:.4f}")

    # Checkpoint every epoch so a Colab disconnect doesn't lose the run
    torch.save({"epoch": epoch + 1, "gen": gen.state_dict(), "disc": disc.state_dict(),
                "opt_gen": opt_gen.state_dict(), "opt_disc": opt_disc.state_dict(),
                "history": history}, CHECKPOINT_PATH)

torch.save(gen.state_dict(), "generator.pth")
print("Saved generator weights to generator.pth")

In [ ]:
epochs_axis = range(1, len(history["d_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_axis, history["d_loss"], label="D loss")
ax1.plot(epochs_axis, history["g_adv"], label="G adversarial loss")
ax1.set_xlabel("Epoch")
ax1.set_title("Adversarial losses")
ax1.legend()
ax2.plot(epochs_axis, history["g_l1"], color="tab:green")
ax2.set_xlabel("Epoch")
ax2.set_title("G L1 loss (colour reconstruction)")
plt.tight_layout()
plt.show()

## 8. Evaluation on the held-out angle blocks

Mean **PSNR** (higher is better, in dB) and **SSIM** (1.0 = identical) between generated and real colour images, both in [0, 1].

In [ ]:
gen.eval()
psnrs, ssims = [], []
with torch.no_grad():
    for gray, real in test_loader:
        fake = gen(gray.to(device))
        for f, r in zip(fake, real):
            f_img, r_img = to_image(f), to_image(r)
            psnrs.append(peak_signal_noise_ratio(r_img, f_img, data_range=1.0))
            ssims.append(structural_similarity(r_img, f_img, data_range=1.0, channel_axis=2))

print(f"Test images: {len(psnrs)}")
print(f"PSNR: {np.mean(psnrs):.2f} dB")
print(f"SSIM: {np.mean(ssims):.4f}")

In [ ]:
# One random sample per species from the test set
samples = test_df.groupby("species").sample(1, random_state=SEED)
sample_ds = FlowerColorizationDataset(samples["path"])

fig, axes = plt.subplots(len(sample_ds), 3, figsize=(7, 2.3 * len(sample_ds)), squeeze=False)
with torch.no_grad():
    for row, (species, (gray, real)) in enumerate(zip(samples["species"], sample_ds)):
        fake = gen(gray.unsqueeze(0).to(device))[0]
        axes[row, 0].imshow(to_image(gray).squeeze(-1), cmap="gray")
        axes[row, 1].imshow(to_image(fake))
        axes[row, 2].imshow(to_image(real))
        axes[row, 0].set_ylabel(species)
for ax, title in zip(axes[0], ["Input (gray)", "Generated", "Real"]):
    ax.set_title(title)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()